# 임베딩 & 벡터 DB 구축

## 전체 파이프라인

```
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[0단계] 세계관 & 소재 입력  (동시 입력)
  경로 A. 세계관 선택
    유형 선택: 만화 / 시리즈 / 영화 / 소설
      → 장르 선택 (스릴러 / 판타지 / 로맨스 … 소설은 공포(호러) 제외)
      → 작품명 검색 또는 직접 입력
    유형 선택: 고전
      → 국가 선택 (한국 / 중국 / 일본)
      → 해당 국가의 장르 선택
      → 작품명 검색 또는 직접 입력
  경로 B. 소재 직접 입력
    키워드 / 아이디어 입력 → 장르 선택 → AI가 세계관 자동 확장
  캐릭터 정보 입력 (소재와 동시 입력)
    ├─ AI 캐릭터 정보: 이름 / 성격 / 외형 / 배경  (유저와 대화할 상대방)
    └─ 유저 페르소나 정보: 이름 / 성격 / 배경      (유저를 대신할 캐릭터)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
          ↓
[핵심 기능 1] AI 시나리오 & 캐릭터 빌더  (1회)
  RAG: 유형/장르/단계별 유사 씬 검색  ← 이 파일이 담당
    scenes 컬렉션  (문화콘텐츠 스토리 데이터, 89,851 docs)
    classics 컬렉션 (동아시아 고전 데이터, 12,717 docs)  ← 고전 선택 시
  → LLM: 두 캐릭터 중심의 기승전결 트리트먼트 생성 (복선·갈등 구조 포함)
  → 시나리오는 유저에게 노출하지 않고 백엔드 진행 가이드로만 사용
  → AI 캐릭터 초기 이미지 생성 → 기준 이미지 저장 (이미지 API 미정)
  → 로어북 자동 초기화 (인물·장소·사건·설정 추출 → Vector DB 색인)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
          ↓
[핵심 기능 2] AI 시나리오 트랜스포머  (1회)
  선형 시나리오 → 분기형 인터랙티브 구조 변환 (Narrative Branching)
  → 분기점(Choice Points) 생성 (갈등 절정부에 선택지 배치)
  → 분기별 전개 방향 설계
  → 엔딩 조건 설계
      해피 엔딩 / 배드 엔딩 / 트루 엔딩 / 루트별 엔딩
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
          ↓
[핵심 기능 3] 인터랙티브 챗 & 플레이  (반복)
  플레이 설정: 세션 옵션 (출력 모델 / 감성 / AI 주도 사건 등)
  매 턴 시스템 프롬프트 구성
    고정 레이어: 유저 페르소나 + 유저 노트 + 감성 설정
    동적 레이어 A — 로어북: 현재 대화 문맥 → Semantic Search
                             → 유사 항목 실시간 주입 (Narrative Consistency Engine)
    동적 레이어 B — RAG 씬: 현재 기승전결 단계 기반 검색  ← 이 파일이 담당
    동적 레이어 C — 요약 기억: 10~15턴 초과 시 자동 요약 주입
  AI 생성: 캐릭터 대화 + 상태창 + 추천 선택지 3개
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
          ↓
[분기점 도달]
  선택지 제시 (2~3개) → 유저 선택 → 분기 상태 저장
  → AI 캐릭터 단계별 이미지 생성 (이미지 API 미정)
      기준 이미지 얼굴 고정 + 현재 분기 상황 묘사
      → 얼굴 일관성 유지, 배경·의상·표정만 변화
  → 해당 분기 시나리오로 대화 계속
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
          ↓
[엔딩 도달]
  엔딩 조건 판단 (호감도 누적 + 선택 플래그 + 사건 달성 여부)
  → 엔딩 씬 생성 (LLM)
  → 엔딩 이미지 생성 (기준 이미지 얼굴 고정 + 엔딩 상황, 이미지 API 미정)
  → 결과 화면: 엔딩 명칭 + 씬 + 이미지 + 분기 경로 + 재도전 버튼
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
```

## 이 파일의 범위

| 단계 | 기능 | 이 파일 | 미구현 |
|------|------|---------|--------|
| 0단계 | 유형/장르/국가 선택 → 검색 파라미터 결정 | ✅ | |
| 핵심 기능 1 | RAG: scenes 컬렉션 구축 및 검색 | ✅ | |
| 핵심 기능 1 | RAG: classics 컬렉션 구축 및 검색 | ✅ | |
| 핵심 기능 1 | RAG 컨텍스트 텍스트 조립 (프롬프트 주입용) | ✅ | |
| 핵심 기능 1 | LLM 기승전결 트리트먼트 생성 | ✅ | |
| 핵심 기능 1 | AI 캐릭터 초기 이미지 생성 (기준 이미지, API 미정) | | ❌ |
| 핵심 기능 1 | 로어북 자동 초기화 (항목 추출 → Vector DB 색인) | | ❌ |
| 핵심 기능 2 | 시나리오 트랜스포머 (선형 → 분기 구조 변환) | | ❌ |
| 핵심 기능 2 | 분기점·엔딩 조건 설계 | | ❌ |
| 핵심 기능 3 | 로어북 Semantic Search & 실시간 컨텍스트 주입 | | ❌ |
| 핵심 기능 3 | 채팅 단계 추적 & 전환 로직 | | ❌ |
| 핵심 기능 3 | 요약 기억 생성 및 주입 | | ❌ |
| 분기점 | 선택지 생성 & 분기 상태 저장 | | ❌ |
| 분기점 | 단계별 이미지 생성 (얼굴 고정, API 미정) | | ❌ |
| 엔딩 | 엔딩 조건 판단 & 씬 생성 & 이미지 생성 | | ❌ |

## 설계 결정 사항

| 항목 | 결정 | 근거 |
|------|------|------|
| **임베딩 모델** | `text-embedding-3-small` | 비용 효율적 ($0.02/1M tokens) |
| **씬 임베딩 텍스트** | `unit_motif + storyline` | 의미적 유사도 검색에 최적화 |
| **고전 임베딩 텍스트** | `주제문` | 단락 내용 요약 1문장 |
| **category_name** | 메타데이터 필터 | 만화/시리즈/영화/소설 유형 구분 |
| **stage** | 메타데이터 필터 | 기승전결 단계별 씬 검색 |
| **causality** | 임베딩 제외, 프롬프트 출력용 | 씬 간 연결 정보 |
| **hero_journey** | framework 필터 없이 유사도에 맡김 | 판타지 맥락 시 자연스럽게 포함 |
| **고전 데이터** | 고전 탭 선택 시에만 추가 검색 | 세계관·분위기 보강 보조 데이터 |
| **이미지 생성 API** | 미정 | Leonardo.ai / Civitai 등 검토 예정 |
| **얼굴 일관성 유지** | 미정 | Character Reference / IP-Adapter 등 검토 예정 |

## 출력 파일

| 경로 | 내용 |
|------|------|
| `vectordb/scenes` | 문화콘텐츠 씬 컬렉션 (89,851개) |
| `vectordb/classics` | 동아시아 고전 단락 컬렉션 (12,717개) |

In [17]:
# 필요 패키지 설치
#!pip install openai chromadb tqdm

In [18]:
import json
import os
import time
from pathlib import Path
from typing import Optional

import chromadb
from openai import OpenAI
from tqdm import tqdm
from dotenv import load_dotenv 

load_dotenv()

# ── 경로 설정 ──────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("C:/Users/User/Desktop/Github/Dive.ai/output")
DB_DIR     = Path("C:/Users/User/Desktop/Github/Dive.ai/vectordb")
DB_DIR.mkdir(exist_ok=True)

# ── OpenAI 클라이언트 ──────────────────────────────────────────────────────────
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

EMBED_MODEL = "text-embedding-3-small"  # 1536차원
BATCH_SIZE  = 500                        # OpenAI API 배치 크기
UPSERT_BATCH = 100                       # ChromaDB 업서트 배치 크기

print(f"OUTPUT_DIR 존재 여부: {OUTPUT_DIR.exists()}")
print(f"DB_DIR: {DB_DIR}")
if os.environ.get("OPENAI_API_KEY"):
    print(f"API Key 로드 완료: {os.environ.get('OPENAI_API_KEY')[:7]}...")
else:
    print("API Key를 찾을 수 없습니다. .env 파일을 확인해주세요.")

OUTPUT_DIR 존재 여부: True
DB_DIR: C:\Users\User\Desktop\Github\Dive.ai\vectordb
API Key 로드 완료: sk-proj...


## 1. 전처리 데이터 로드

`문화콘텐츠,고전_전처리.ipynb` 실행 결과로 생성된 JSON 파일을 로드합니다.

In [19]:
with open(OUTPUT_DIR / "scene_metadata_index.json", encoding="utf-8") as f:
    scene_index = json.load(f)

with open(OUTPUT_DIR / "classic_paragraph_index.json", encoding="utf-8") as f:
    classic_index = json.load(f)

print(f"문화콘텐츠 씬:   {len(scene_index):,}개")
print(f"동아시아 고전 단락: {len(classic_index):,}개")

문화콘텐츠 씬:   89,851개
동아시아 고전 단락: 12,717개


## 2. 임베딩 텍스트 구성

### 씬 임베딩 텍스트
- `unit_motif + storyline` 조합
- `unit_motif` 누락 시 `stage`로 fallback (7.4% 누락 대응)
- `causality`는 임베딩 제외 — 씬 자체 내용이 아닌 다음 씬으로의 연결 정보

### 고전 단락 임베딩 텍스트
- `주제문` 단독 사용 — 단락 내용을 한 문장으로 요약한 필드

In [20]:
def build_scene_text(scene: dict) -> str:
    motif     = scene.get("unit_motif") or scene.get("stage", "")
    storyline = scene.get("storyline", "")
    return f"{motif} {storyline}".strip()


def build_classic_text(para: dict) -> str:
    return para.get("summary", "")


# ── 임베딩 비용 예측 ───────────────────────────────────────────────────────────
AVG_SCENE_TOKENS   = 35   # unit_motif + storyline 평균 토큰 수 추정
AVG_CLASSIC_TOKENS = 45   # 주제문 평균 토큰 수 추정
COST_PER_1M        = 0.02 # text-embedding-3-small (USD)

total_tokens = len(scene_index) * AVG_SCENE_TOKENS + len(classic_index) * AVG_CLASSIC_TOKENS
estimated_cost = total_tokens / 1_000_000 * COST_PER_1M

print(f"예상 총 토큰:  {total_tokens:,}")
print(f"예상 비용:     ${estimated_cost:.4f} (text-embedding-3-small 기준)")
print()

# 샘플 확인
sample = scene_index[0]
print("[씬 임베딩 텍스트 샘플]")
print(build_scene_text(sample))
print()
print("[고전 임베딩 텍스트 샘플]")
print(build_classic_text(classic_index[0]))

예상 총 토큰:  3,717,050
예상 비용:     $0.0743 (text-embedding-3-small 기준)

[씬 임베딩 텍스트 샘플]
긴장감 C001은 지하철 안에서 졸다가 깨어 내리는데 두 여자아이를 발견하고 섬찟한다.

[고전 임베딩 텍스트 샘플]
연왕후가 모란을 죽이기 위해 궁녀를 병사로 삼겠다고 하고 상께서 궁녀를 모았지만 그 중에 모란이 보이지 않았다.


## 3. ChromaDB 컬렉션 초기화

- `scenes`: 문화콘텐츠 씬 컬렉션
- `classics`: 동아시아 고전 단락 컬렉션
- 거리 함수: cosine (유사도 = 1 - distance)
- PersistentClient: `vectordb/` 폴더에 영구 저장

In [21]:
chroma_client = chromadb.PersistentClient(path=str(DB_DIR))

scene_collection = chroma_client.get_or_create_collection(
    name="scenes",
    metadata={"hnsw:space": "cosine"},
)

classic_collection = chroma_client.get_or_create_collection(
    name="classics",
    metadata={"hnsw:space": "cosine"},
)

print(f"scenes  컬렉션 현재 항목 수: {scene_collection.count():,}")
print(f"classics 컬렉션 현재 항목 수: {classic_collection.count():,}")

scenes  컬렉션 현재 항목 수: 89,851
classics 컬렉션 현재 항목 수: 12,717


## 4. 문화콘텐츠 씬 임베딩 및 저장

### 저장 메타데이터

| 필드 | 용도 |
|------|------|
| `work_id` | 작품 식별자 |
| `category_name` | 유형 필터 (만화/시리즈/영화/소설) |
| `stage` | 기승전결 파트 필터링 |
| `framework` | storyhelper / hero_journey 필터링 |
| `genre` | 장르 post-filtering (콤마 구분 문자열) |
| `theme` | 작품 테마 (출력 컨텍스트) |
| `unit_motif` | 검색 결과 출력 컨텍스트 |
| `work_motif` | 작품 모티프 참고 |
| `main_character` | 주인공 유형 참고 |
| `storyline` | LLM 프롬프트 주입 텍스트 |
| `causality` | 있을 때만 프롬프트에 포함 (씬 간 인과 힌트) |
| `dominant_emotions` | 저장만, 현재 미사용 |

In [22]:
def embed_and_store_scenes():
    current_count = scene_collection.count()
    total_count   = len(scene_index)

    if current_count >= total_count:
        print(f"이미 완료: {current_count:,}개 씬 저장됨. 스킵합니다.")
        return

    print("기존 저장 ID 확인 중...")
    existing_ids = set()
    if current_count > 0:
        result = scene_collection.get(include=[])
        existing_ids = set(result["ids"])

    to_embed = [s for s in scene_index if s["scene_id"] not in existing_ids]
    print(f"임베딩 대상: {len(to_embed):,}개 / 전체 {total_count:,}개")

    if not to_embed:
        print("임베딩할 씬이 없습니다.")
        return

    texts = [build_scene_text(s) for s in to_embed]
    embeddings = []

    print("OpenAI 임베딩 중...")
    for i in tqdm(range(0, len(texts), BATCH_SIZE)):
        batch = texts[i:i + BATCH_SIZE]
        response = client.embeddings.create(model=EMBED_MODEL, input=batch)
        embeddings.extend([r.embedding for r in response.data])
        time.sleep(0.05)

    print("ChromaDB 저장 중...")
    for i in tqdm(range(0, len(to_embed), UPSERT_BATCH)):
        batch_scenes     = to_embed[i:i + UPSERT_BATCH]
        batch_embeddings = embeddings[i:i + UPSERT_BATCH]

        scene_collection.upsert(
            ids=[s["scene_id"] for s in batch_scenes],
            embeddings=batch_embeddings,
            documents=[build_scene_text(s) for s in batch_scenes],
            metadatas=[
                {
                    "work_id":           s.get("work_id", ""),
                    "category_name":     s.get("category_name", ""),   # 만화/시리즈/영화/소설
                    "genre":             ",".join(s.get("genre", [])),
                    "stage":             s.get("stage", ""),
                    "framework":         s.get("framework", "storyhelper"),
                    "theme":             s.get("theme", ""),
                    "unit_motif":        s.get("unit_motif") or s.get("stage", ""),
                    "work_motif":        s.get("work_motif", ""),
                    "main_character":    s.get("main_character", ""),
                    "storyline":         (s.get("storyline") or "")[:500],
                    "causality":         (s.get("causality") or "")[:300],
                    "dominant_emotions": ",".join(s.get("dominant_emotions", [])),
                }
                for s in batch_scenes
            ],
        )

    print(f"완료. 총 {scene_collection.count():,}개 씬 저장됨.")


embed_and_store_scenes()

이미 완료: 89,851개 씬 저장됨. 스킵합니다.


## 5. 동아시아 고전 단락 임베딩 및 저장

고전 데이터는 **고전 배경 선택 시에만** 세계관·분위기 보강용으로 참조합니다.  
`주제문` 한 문장을 임베딩하여 유저 입력과의 의미적 유사도를 계산합니다.

In [23]:
# 기존에 잘못 저장된 (메타데이터가 빈) 고전 컬렉션을 삭제합니다.
chroma_client.delete_collection("classics")
# 다시 새로 만듭니다.
classic_collection = chroma_client.get_or_create_collection(
name="classics",
metadata={"hnsw:space": "cosine"},
)


def embed_and_store_classics():
    current_count = classic_collection.count()
    total_count   = len(classic_index)

    if current_count >= total_count:
        print(f"이미 완료: {current_count:,}개 단락 저장됨. 스킵합니다.")
        return

    print("기존 저장 ID 확인 중...")
    existing_ids = set()
    if current_count > 0:
        result = classic_collection.get(include=[])
        existing_ids = set(result["ids"])

    to_embed = [p for p in classic_index if p["paragraph_id"] not in existing_ids]
    print(f"임베딩 대상: {len(to_embed):,}개 / 전체 {total_count:,}개")

    if not to_embed:
        print("임베딩할 단락이 없습니다.")
        return

    texts = [build_classic_text(p) for p in to_embed]
    embeddings = []

    print("OpenAI 임베딩 중...")
    for i in tqdm(range(0, len(texts), BATCH_SIZE)):
        batch = texts[i:i + BATCH_SIZE]
        response = client.embeddings.create(model=EMBED_MODEL, input=batch)
        embeddings.extend([r.embedding for r in response.data])
        time.sleep(0.05)

    print("ChromaDB 저장 중...")
    for i in tqdm(range(0, len(to_embed), UPSERT_BATCH)):
        batch_paras      = to_embed[i:i + UPSERT_BATCH]
        batch_embeddings = embeddings[i:i + UPSERT_BATCH]

        classic_collection.upsert(
            ids=[p["paragraph_id"] for p in batch_paras],
            embeddings=batch_embeddings,
            documents=[build_classic_text(p) for p in batch_paras],
            metadatas=[
                {
                    "work_id":   p.get("work_id", ""),
                    "country":   p.get("country", ""),
                    "genre":     ",".join(p.get("genre", [])),
                    "motif":     ",".join(p.get("motif", [])),
                    "space":     ",".join(p.get("space", [])),
                    "character": ",".join(p.get("character", [])),
                    "summary":    (p.get("summary") or "")[:500],
                }
                for p in batch_paras
            ],
        )

    print(f"완료. 총 {classic_collection.count():,}개 고전 단락 저장됨.")


embed_and_store_classics()

기존 저장 ID 확인 중...
임베딩 대상: 12,717개 / 전체 12,717개
OpenAI 임베딩 중...


100%|██████████| 26/26 [00:57<00:00,  2.21s/it]


ChromaDB 저장 중...


100%|██████████| 128/128 [00:36<00:00,  3.52it/s]

완료. 총 12,717개 고전 단락 저장됨.


## 6. 기승전결 Stage 매핑

스토리헬퍼 16단계를 기승전결 4구간으로 묶어 `story_arc` 파라미터로 검색할 수 있도록 합니다.  
stage는 임베딩에 포함되지 않고 **메타데이터 post-filtering**에만 사용됩니다.

In [24]:
# 스토리헬퍼 16단계 → 기승전결 매핑
STORY_ARC_STAGES: dict[str, list[str]] = {
    "기": [
        "Opening Salvo",    # 첫 장면, 세계관 제시
        "Main Character",   # 주인공 소개
        "Setting-up",       # 배경 및 관계 설정
    ],
    "승": [
        "1st Accident",     # 첫 번째 사건
        "Villains Move",    # 적대자 등장
        "Doubts & Debate",  # 주인공 갈등
        "Making a Choice",  # 결단
        "Choice to Fight",  # 싸우기로 결심
    ],
    "전": [
        "Ups & Downs",      # 갈등 반복
        "2nd Accident",     # 두 번째 위기
        "Innermost Cave",   # 최대 위험
        "Defeat",           # 패배/최대 위기
        "Resurrection",     # 재기
        "Another Story",    # 서브플롯
    ],
    "결": [
        "Trailer Moments",  # 클라이맥스 직전
        "Final Salvo",      # 최종 해결
    ],
}

for arc, stages in STORY_ARC_STAGES.items():
    print(f"{arc}: {', '.join(stages)}")

기: Opening Salvo, Main Character, Setting-up
승: 1st Accident, Villains Move, Doubts & Debate, Making a Choice, Choice to Fight
전: Ups & Downs, 2nd Accident, Innermost Cave, Defeat, Resurrection, Another Story
결: Trailer Moments, Final Salvo


## 7. 검색 함수

### retrieve_scenes
- 유저 입력을 임베딩 → 유사 씬 검색
- `genre`, `story_arc`, `stage`, `framework` 필터 지원
- genre는 ChromaDB 메타데이터 substring 미지원으로 **Python post-filtering**
- `framework=None`: storyhelper + hero_journey 전체 검색 (판타지 기본)

### retrieve_classics
- 고전 배경 선택 시에만 호출
- `country` 필터 + genre/motif post-filtering

In [29]:
def retrieve_scenes(
    query: str,
    category: Optional[str] = None,
    genre: Optional[str] = None,
    stage: Optional[str] = None,
    story_arc: Optional[str] = None,
    framework: Optional[str] = None,
    top_k: int = 5,
) -> list[dict]:
    query_vec = client.embeddings.create(
        model=EMBED_MODEL, input=[query]
    ).data[0].embedding

    # --- [에러 해결 포인트] ChromaDB 복합 필터($and) 구성 ---
    filter_list = []

    if framework:
        filter_list.append({"framework": {"$eq": framework}})
    if category:
        filter_list.append({"category_name": {"$eq": category}})

    # 기승전결(story_arc) 단계 설정
    arc_stages = STORY_ARC_STAGES.get(story_arc, []) if story_arc else []
    if stage:
        filter_list.append({"stage": {"$eq": stage}})
    elif arc_stages:
        filter_list.append({"stage": {"$in": arc_stages}})

    # 필터가 여러 개일 경우 $and로 묶고, 하나일 경우 그대로 사용
    if len(filter_list) > 1:
        where = {"$and": filter_list}
    elif len(filter_list) == 1:
        where = filter_list[0]
    else:
        where = None
    # -----------------------------------------------------

    fetch_k = top_k * 15 if genre else top_k
    fetch_k = min(fetch_k, scene_collection.count())

    query_params: dict = {
        "query_embeddings": [query_vec],
        "n_results":        fetch_k,
        "include":          ["metadatas", "documents", "distances"],
    }
    if where:
        query_params["where"] = where

    results = scene_collection.query(**query_params)

    scenes = []
    for meta, doc, dist in zip(
        results["metadatas"][0],
        results["documents"][0],
        results["distances"][0],
    ):
        if genre and genre not in meta.get("genre", ""):
            continue

        scenes.append({
            "score":         round(1 - dist, 4),
            "category":      meta.get("category_name", ""),
            "stage":         meta.get("stage", ""),
            "framework":     meta.get("framework", ""),
            "unit_motif":    meta.get("unit_motif", ""),
            "genre":         meta.get("genre", ""),
            "storyline":     meta.get("storyline", ""),
            "causality":     meta.get("causality", ""),
        })
        if len(scenes) >= top_k:
            break

    return scenes


print("검색 함수 정의 완료.")

검색 함수 정의 완료.


## 8. RAG 프롬프트 빌더

검색 결과를 LLM 프롬프트 주입용 텍스트로 변환합니다.

### 유형별 선택 가능 장르 (`CATEGORY_TO_GENRES`)

소설 카테고리에 `공포(호러)` 데이터가 없으므로, 소설 선택 시 해당 장르를 프론트엔드에서 노출하지 않습니다.

| 유형 | 장르 선택지 |
|------|------------|
| 영화 | 드라마 · 멜로/로맨스 · 스릴러 · 판타지 · 액션 · 미스터리 · 코미디 · SF · 공포(호러) · 전쟁 |
| 시리즈 | 드라마 · 멜로/로맨스 · 스릴러 · 판타지 · 액션 · 미스터리 · 코미디 · SF · 공포(호러) · 전쟁 |
| 소설 | 드라마 · 멜로/로맨스 · 스릴러 · 판타지 · 액션 · 미스터리 · 코미디 · SF · 전쟁 **(공포(호러) 제외)** |
| 만화 | 드라마 · 멜로/로맨스 · 스릴러 · 판타지 · 액션 · 미스터리 · 코미디 · SF · 공포(호러) · 전쟁 |

### 고전 탭 UI → 백엔드 파라미터 흐름

```
[프론트엔드]                         [백엔드]
장르 탭 선택
  └─ 일반 장르 선택 (스릴러 등)  →  genre="스릴러", country=None
  └─ 고전 탭
       └─ 한국 선택              →  country="한국"
            └─ 가문소설 선택     →  classic_genre="가문소설"
            └─ (미선택)          →  classic_genre=None  ← 국가 전체 검색
       └─ 중국 선택              →  country="중국"
       └─ 일본 선택              →  country="일본"
```

### 국가별 선택 가능 장르 (`COUNTRY_TO_GENRES`)

| 국가 | 장르 선택지 |
|------|------------|
| 한국 | 가문소설 · 판타지 · 로맨스 · 영웅소설 · 미스터리 · 호러 |
| 중국 | 판타지 · 로맨스 · 무협 · 호러 · 미스터리 |
| 일본 | 설화 · 미스터리 · 호러 · 편지소설 · 영험담 |

### 동작 방식
- `country=None` → 고전 단락 검색 안 함 (일반 씬만 사용)
- `classic_genre=None` → 해당 국가 전체 장르 대상으로 검색
- `classic_genre="설화"` → 해당 장르 단일 필터로 검색
- `causality`: 있을 때만 씬 컨텍스트에 포함 — 씬 간 흐름 힌트

In [26]:
# ── 유형별 장르 목록 (프론트엔드 선택지 관리) ─────────────────────────────────
# 소설 카테고리에 공포(호러) 데이터 없음 → 소설 선택 시 해당 장르 노출 안 함
CATEGORY_TO_GENRES: dict[str, list[str]] = {
    "영화":   ["드라마", "멜로/로맨스", "스릴러", "판타지", "액션", "미스터리", "코미디", "SF", "공포(호러)", "전쟁"],
    "시리즈": ["드라마", "멜로/로맨스", "스릴러", "판타지", "액션", "미스터리", "코미디", "SF", "공포(호러)", "전쟁"],
    "소설":   ["드라마", "멜로/로맨스", "스릴러", "판타지", "액션", "미스터리", "코미디", "SF", "전쟁"],
    "만화":   ["드라마", "멜로/로맨스", "스릴러", "판타지", "액션", "미스터리", "코미디", "SF", "공포(호러)", "전쟁"],
}

# ── 국가별 고전 장르 목록 (프론트엔드 선택지 + 백엔드 필터 공유) ───────────────
# 실제 데이터 기준 (동아시아 고전.ipynb 섹션 6 참고)
# 한국: 가문소설(3,545) · 판타지(2,655) · 로맨스(1,747) · 영웅소설(380) 등
# 중국: 판타지(1,137) · 로맨스(204) · 무협(194) · 호러(149) 등
# 일본: 설화(2,980) · 미스터리(235) · 호러(82) · 편지소설(72) 등

COUNTRY_TO_GENRES: dict[str, list[str]] = {
    "한국": ["가문소설", "판타지", "로맨스", "영웅소설", "미스터리", "호러"],
    "중국": ["판타지", "로맨스", "무협", "호러", "미스터리"],
    "일본": ["설화", "미스터리", "호러", "편지소설", "영험담"],
}

CLASSIC_COUNTRIES: set[str] = set(COUNTRY_TO_GENRES.keys())  # {"한국", "중국", "일본"}


def build_scene_context(scenes: list[dict]) -> str:
    if not scenes:
        return ""
    lines = ["[유사 작품 씬 패턴 참고]"]
    for s in scenes:
        tag  = f"{s['stage']} / {s['unit_motif']}" if s["unit_motif"] else s["stage"]
        line = f"- [{tag}] {s['storyline']}"
        if s.get("causality"):
            line += f"\n  → {s['causality']}"
        lines.append(line)
    return "\n".join(lines)


def build_classic_context(paragraphs: list[dict]) -> str:
    if not paragraphs:
        return ""
    lines = ["[동아시아 고전 참고 — 세계관·분위기]"]
    for p in paragraphs:
        motif_str = p["motif"][:40] if p["motif"] else "?"
        space_str = p["space"][:20] if p["space"] else "?"
        lines.append(f"- [{p['country']} / {motif_str} / {space_str}] {p['summary']}")
    return "\n".join(lines)


def build_rag_context(
    query: str,
    category: Optional[str] = None,
    genre: Optional[str] = None,
    story_arc: Optional[str] = None,
    country: Optional[str] = None,
    classic_genre: Optional[str] = None,
    top_k_scenes: int = 5,
    top_k_classics: int = 3,
) -> str:
    """씬 + 고전 RAG 컨텍스트 통합 생성.

    Parameters
    ----------
    query         : 유저 입력 소재/맥락
    category      : 유형 ('만화' | '시리즈' | '영화' | '소설') — 고전 탭이면 None
    genre         : 장르 (예: '스릴러') — CATEGORY_TO_GENRES 기준으로 프론트에서 검증됨
    story_arc     : 기승전결 파트 (예: '결')
    country       : 고전 탭 선택 국가 ('한국' | '중국' | '일본') — None이면 고전 미사용
    classic_genre : 국가 내 선택 장르 (예: '가문소설') — None이면 국가 전체 검색
    top_k_scenes  : 씬 검색 수
    top_k_classics: 고전 단락 검색 수
    """
    # 씬 검색 — framework=None: storyhelper + hero_journey 전체 (유사도에 맡김)
    scenes  = retrieve_scenes(
        query, category=category, genre=genre, story_arc=story_arc, top_k=top_k_scenes
    )
    context = build_scene_context(scenes)

    # 고전 탭 선택 시에만 고전 단락 추가
    if country and country in CLASSIC_COUNTRIES:
        genre_keywords = [classic_genre] if classic_genre else COUNTRY_TO_GENRES.get(country)
        classics = retrieve_classics(
            query,
            country=country,
            genre_keywords=genre_keywords,
            top_k=top_k_classics,
        )
        if classics:
            context += "\n\n" + build_classic_context(classics)

    return context


print("RAG 프롬프트 빌더 정의 완료.")
print()
print("CATEGORY_TO_GENRES:")
for cat, genres in CATEGORY_TO_GENRES.items():
    print(f"  {cat}: {', '.join(genres)}")
print()
print("COUNTRY_TO_GENRES:")
for c, genres in COUNTRY_TO_GENRES.items():
    print(f"  {c}: {', '.join(genres)}")

RAG 프롬프트 빌더 정의 완료.

CATEGORY_TO_GENRES:
  영화: 드라마, 멜로/로맨스, 스릴러, 판타지, 액션, 미스터리, 코미디, SF, 공포(호러), 전쟁
  시리즈: 드라마, 멜로/로맨스, 스릴러, 판타지, 액션, 미스터리, 코미디, SF, 공포(호러), 전쟁
  소설: 드라마, 멜로/로맨스, 스릴러, 판타지, 액션, 미스터리, 코미디, SF, 전쟁
  만화: 드라마, 멜로/로맨스, 스릴러, 판타지, 액션, 미스터리, 코미디, SF, 공포(호러), 전쟁

COUNTRY_TO_GENRES:
  한국: 가문소설, 판타지, 로맨스, 영웅소설, 미스터리, 호러
  중국: 판타지, 로맨스, 무협, 호러, 미스터리
  일본: 설화, 미스터리, 호러, 편지소설, 영험담


## 9. 검색 테스트

임베딩 완료 후 아래 셀을 실행하여 검색 품질을 확인합니다.

In [27]:
SEP = "=" * 60

# ── 테스트 1: 스릴러 — 승 파트 (고전 미선택) ──────────────────────────────────
print(SEP)
print("테스트 1: 스릴러 — 승 파트 (고전 미선택)")
print(SEP)
context = build_rag_context(
    query="형사가 사건의 단서를 쫓으며 위험에 빠진다",
    genre="스릴러",
    story_arc="승",
)
print(context)

print()

# ── 테스트 2: 판타지 — 결 파트 (고전 미선택) ──────────────────────────────────
print(SEP)
print("테스트 2: 판타지 — 결 파트 (고전 미선택)")
print(SEP)
context = build_rag_context(
    query="모든 시련을 이겨낸 영웅이 고향으로 돌아온다",
    genre="판타지",
    story_arc="결",
)
print(context)

print()

# ── 테스트 3: 고전 탭 → 한국 → 가문소설 ─────────────────────────────────────
print(SEP)
print("테스트 3: 고전 탭 → 한국 → 가문소설")
print(SEP)
context = build_rag_context(
    query="두 가문의 갈등 속에서 피어나는 비극적 사랑 이야기",
    genre="로맨스",
    story_arc="전",
    country="한국",
    classic_genre="가문소설",
)
print(context)

print()

# ── 테스트 4: 고전 탭 → 중국 → 무협 ─────────────────────────────────────────
print(SEP)
print("테스트 4: 고전 탭 → 중국 → 무협")
print(SEP)
context = build_rag_context(
    query="강호를 떠돌며 복수를 꿈꾸는 무사의 여정",
    genre="판타지",
    story_arc="승",
    country="중국",
    classic_genre="무협",
)
print(context)

print()

# ── 테스트 5: 고전 탭 → 일본 → 설화 ─────────────────────────────────────────
print(SEP)
print("테스트 5: 고전 탭 → 일본 → 설화")
print(SEP)
context = build_rag_context(
    query="요괴를 물리치고 마을을 구하는 승려의 이야기",
    genre="판타지",
    story_arc="결",
    country="일본",
    classic_genre="설화",
)
print(context)

print()

# ── 테스트 6: 고전 탭 → 한국 (장르 미선택 — 국가 전체 검색) ──────────────────
print(SEP)
print("테스트 6: 고전 탭 → 한국 (장르 미선택)")
print(SEP)
context = build_rag_context(
    query="조선 시대 신분을 초월한 사랑과 시련",
    story_arc="기",
    country="한국",
    classic_genre=None,  # 국가 전체 장르 검색
)
print(context)

테스트 1: 스릴러 — 승 파트 (고전 미선택)
[유사 작품 씬 패턴 참고]
- [1st Accident / 상황 설명] 이 형사가 피해자들의 신원조회 내용을 보고한다.
  → Seeker(추적/추구)
- [Making a Choice / 감정을 부정] C002은 너무 힘들어 자살을 시도하고 판사는 C003에게 사형을 선고한다.
  → Lack(결핍)
- [1st Accident / 신분위장] 대신 안전하게 지켜주겠다며 형사는 수녀원에서 두 달만 숨어 있으라고 한다.
  → Crisis of the heart(지극한 슬픔)
- [Making a Choice / 강제된 과업] C001가 고용한 탐정이 죽고 위험에 빠진다.
  → Facing the greatest fear(심연을 직면함)
- [Doubts & Debate / 사고사] 소방대원들이 죽어나가고, 하객들도 위험에 처한다.
  → Lack(결핍)

테스트 2: 판타지 — 결 파트 (고전 미선택)
[유사 작품 씬 패턴 참고]
- [Final Salvo / 덜떨어진 영웅] C018는 마을의 영웅이 되고 주민들이 무너진 C012의 집을 복구해준다.
  → Courtship(구애)
- [Final Salvo / Final Salvo] 모두들 현실로 돌아와서 각자 돌아가지만 놀이기구에서의 기억들은 다들 아름답게 가져간다.
- [Final Salvo / 이별] C001는 C012과 야생마들을 자연으로 돌려보낸다.
  → Return(귀환)
- [Final Salvo / 시련] 생명력이 바닥난 C001는 몰려오는 적들을 보고 죽음을 받아들인다.
  → Last test(마지막 시험/시련)
- [Final Salvo / 해피엔딩] 평온을 되찾은 탐사대원들은 후손을 이어가며 항해를 계속한다.
  → Return(귀환)

테스트 3: 고전 탭 → 한국 → 가문소설
[유사 작품 씬 패턴 참고]
- [Resurrection / 광기어린 사랑] 두 사람이 배를 향해 헤엄쳐간다.
  → Setback(후퇴/역행/좌절)
- [A

## 10. LLM모델 테스트(GPT-4o-mini)

LLM모델에게 시나리오 작성 프롬프트를 넣어 실제로 어떻게 작동하는지 확인합니다.

In [31]:
def generate_scenario_with_rag(
    user_query: str,
    genre: str = "판타지",
    category: str = "시리즈",
    country: Optional[str] = None,
    classic_genre: Optional[str] = None
):
    # [RAG 단계]
    rag_context = build_rag_context(
        query=user_query,
        category=category,
        genre=genre,
        story_arc="기",
        top_k_scenes=5
    )

    # [프롬프트 보강] - 구조를 강제하고 잡설을 빼도록 지시
    system_prompt = f"""
너는 군더더기 없이 핵심만 짚어주는 전문 시나리오 작가야.
제공된 소재를 바탕으로 {category}용 {genre} 시나리오의 '기승전결' 뼈대를 작성해.

[작성 원칙]
1. 반드시 #### 기, #### 승, #### 전, #### 결 4단계를 명확히 구분할 것.
2. '전'은 갈등의 최고조(Climax)를, '결'은 사건의 해결과 후일담(Ending)을 담을 것.
3. 소설이나 영화 소개글 같은 '이 시나리오는~' 식의 설명이나 맺음말은 절대 하지 마.
4. 오직 시나리오의 구조적 내용만 출력할 것.
5. 등장인물은 C001, C002 형식을 유지하면서 이름(역할)을 병기할 것.
"""

    user_prompt = f"""
[사용자 소재]
{user_query}

[참고 패턴]
{rag_context}

위 내용을 바탕으로 시나리오 뼈대를 작성해줘.
특히 '결' 단계는 유저의 선택에 따른 '멀티 엔딩'의 가능성을 암시하며 마무리해줘.
"""

    print(f"--- '{category}/{genre}' 시나리오 생성 중 ---")
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.7,
        max_tokens=1500
    )

    return response.choices[0].message.content

# 다시 실행 테스트
test_query = "세상 모든 마력을 흡수하는 저주받은 소녀와 그녀를 지키는 몰락 기사의 여정"
final_scenario = generate_scenario_with_rag(user_query=test_query, genre="판타지", category="소설")

print("\n[수정된 시나리오 뼈대]\n")
print(final_scenario)

--- '소설/판타지' 시나리오 생성 중 ---

[수정된 시나리오 뼈대]

#### 기
C001 (몰락 기사)은 C002 (저주받은 소녀)를 우연히 발견하고 그녀를 보호하기로 결심한다. C002는 세상 모든 마력을 흡수하는 저주를 지닌 존재로, 그녀의 능력 때문에 여러 세력의 위협을 받는다. C001은 C002를 안전한 장소로 옮기기 위해 과거의 동료들을 찾아가고, 그 과정에서 두 사람의 인연이 시작된다.

#### 승
C001과 C002는 함께 여행을 하며 서로의 상처를 나누고, C002의 저주를 풀기 위한 방법을 찾기 시작한다. 이들은 다양한 시련을 겪으며, C001의 옛 동료들인 C003 (음유시인)과 C004 (마법사)와 힘을 합친다. 그러나 저주를 풀기 위한 열쇠를 쥐고 있는 강력한 적, C005 (암흑 마법사)가 나타나 이들을 위협한다. 감정이 고조되며, C002는 자신의 저주로 인한 고통을 자각하고 C001은 그녀를 지키기 위해 더욱 결연한 마음을 다짐한다.

#### 전
C005가 C002를 붙잡으려 하자, C001은 목숨을 걸고 싸운다. 하지만 전투 중 C002의 저주가 발동되어 주변의 마력이 급증하고, C005는 그 힘을 이용해 C002를 완전히 지배하려 한다. C001은 C002를 구하기 위해 저주를 풀 방법을 찾아야 하는 절체절명의 순간에 도달한다. C002는 자신의 존재가 모두에게 위협이 될 수 있음을 깨닫고, C001과의 결단을 내리는 갈림길에 서게 된다.

#### 결
C001이 C002를 구하고 저주를 풀기 위한 결정을 내리자, C002는 자신의 힘을 받아들이기로 결심한다. 그러나 저주가 완전히 사라지지 않아 그녀는 여전히 마력을 흡수할 위험에 처해 있다. 이 과정에서 C001은 C002와 함께 새로운 삶을 시작할 방법을 모색하거나, C002가 저주를 받아들이고 세상의 마력을 지키기로 결심하는 선택지를 제공한다. 두 사람의 선택에 따라 서로의 운명이 달라지며, 다양한 엔딩이 펼쳐질 가능성이 열려 있다.


In [32]:
def generate_scenario_with_rag(
    user_query: str,
    genre: str = "판타지",
    category: str = "시리즈",
    country: Optional[str] = None,
    classic_genre: Optional[str] = None
):
    # [RAG 단계]
    rag_context = build_rag_context(
        query=user_query,
        category=category,
        genre=genre,
        story_arc="기",
        top_k_scenes=5
    )

    # [프롬프트 보강] - 구조를 강제하고 잡설을 빼도록 지시
    system_prompt = f"""
너는 군더더기 없이 핵심만 짚어주는 전문 시나리오 작가야.
제공된 소재를 바탕으로 {category}용 {genre} 시나리오의 '기승전결' 뼈대를 작성해.

[작성 원칙]
1. 반드시 #### 기, #### 승, #### 전, #### 결 4단계를 명확히 구분할 것.
2. '전'은 갈등의 최고조(Climax)를, '결'은 사건의 해결과 후일담(Ending)을 담을 것.
3. 소설이나 영화 소개글 같은 '이 시나리오는~' 식의 설명이나 맺음말은 절대 하지 마.
4. 오직 시나리오의 구조적 내용만 출력할 것.
5. 등장인물은 C001, C002 형식을 유지하면서 이름(역할)을 병기할 것.
"""

    user_prompt = f"""
[사용자 소재]
{user_query}

[참고 패턴]
{rag_context}

위 내용을 바탕으로 시나리오 뼈대를 작성해줘.
특히 '결' 단계는 유저의 선택에 따른 '멀티 엔딩'의 가능성을 암시하며 마무리해줘.
"""

    print(f"--- '{category}/{genre}' 시나리오 생성 중 ---")
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.7,
        max_tokens=1500
    )

    return response.choices[0].message.content

# 다시 실행 테스트
test_query = "기억을 사고파는 편의점과 그곳의 아르바이트생"
final_scenario = generate_scenario_with_rag(user_query=test_query, genre="판타지", category="시리즈")

print("\n[수정된 시나리오 뼈대]\n")
print(final_scenario)

--- '시리즈/판타지' 시나리오 생성 중 ---

[수정된 시나리오 뼈대]

#### 기
C001(아르바이트생)은 C002(편의점 점장)과 함께 기억을 사고파는 편의점에서 일하고 있다. 어느 날, C001은 편의점의 신비한 고객들이 과거의 기억을 구매하는 모습을 목격하고 호기심을 느낀다. 

#### 승
C001은 자신의 잃어버린 기억을 찾기 위해 C002에게 도움을 요청한다. C002는 처음에는 망설이지만, C001의 결심을 보며 기억 회복에 대한 비밀을 털어놓는다. 둘은 기억의 열쇠가 편의점의 한 특정한 물건인 것을 알게 된다.

#### 전
C001과 C002는 그 물건을 찾기 위해 편의점의 깊은 곳으로 들어가지만, 그곳에서 기억을 잃은 악당 C003(고객)과 마주친다. C003은 기억을 거래하며 잃어버린 과거를 찾으려 하고, C001과 C002는 자신의 기억과 C003의 기억이 얽혀있음을 발견하게 된다. 갈등이 최고조에 달하며 세 사람은 서로의 기억을 되찾기 위한 치열한 대결을 벌인다.

#### 결
C001은 C003의 기억을 통해 자신의 과거를 회복하고, C002는 C001을 돕기 위해 자신의 기억을 희생할 준비를 한다. 마지막 순간, C001은 C003에게 기억을 되팔기 위한 선택을 하게 되며, 이에 따라 C003의 과거가 밝혀진다. 선택에 따라 C001은 C002와 함께 기억을 지키거나, C003의 기억을 되찾아주고 새로운 시작을 맞이할 수 있는 멀티 엔딩의 가능성이 열린다.


In [33]:
def generate_scenario_with_rag(
    user_query: str,
    genre: str = "액션",
    category: str = "시리즈",
    country: Optional[str] = None,
    classic_genre: Optional[str] = None
):
    # [RAG 단계]
    rag_context = build_rag_context(
        query=user_query,
        category=category,
        genre=genre,
        story_arc="기",
        top_k_scenes=5
    )

    # [프롬프트 보강] - 구조를 강제하고 잡설을 빼도록 지시
    system_prompt = f"""
너는 군더더기 없이 핵심만 짚어주는 전문 시나리오 작가야.
제공된 소재를 바탕으로 {category}용 {genre} 시나리오의 '기승전결' 뼈대를 작성해.

[작성 원칙]
1. 반드시 #### 기, #### 승, #### 전, #### 결 4단계를 명확히 구분할 것.
2. '전'은 갈등의 최고조(Climax)를, '결'은 사건의 해결과 후일담(Ending)을 담을 것.
3. 소설이나 영화 소개글 같은 '이 시나리오는~' 식의 설명이나 맺음말은 절대 하지 마.
4. 오직 시나리오의 구조적 내용만 출력할 것.
5. 등장인물은 C001, C002 형식을 유지하면서 이름(역할)을 병기할 것.
"""

    user_prompt = f"""
[사용자 소재]
{user_query}

[참고 패턴]
{rag_context}

위 내용을 바탕으로 시나리오 뼈대를 작성해줘.
특히 '결' 단계는 유저의 선택에 따른 '멀티 엔딩'의 가능성을 암시하며 마무리해줘.
"""

    print(f"--- '{category}/{genre}' 시나리오 생성 중 ---")
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.7,
        max_tokens=1500
    )

    return response.choices[0].message.content

# 다시 실행 테스트
test_query = "조선시대판 좀비 사태와 이를 막으려는 세자"
final_scenario = generate_scenario_with_rag(user_query=test_query, genre="액션", category="만화")

print("\n[수정된 시나리오 뼈대]\n")
print(final_scenario)

--- '만화/액션' 시나리오 생성 중 ---

[수정된 시나리오 뼈대]

#### 기  
C001 (세자)은 조선시대에 평화로운 삶을 살고 있다. 그러나, 갑자기 마을에서 좀비가 나타나 사람들을 공격하기 시작한다. C001은 궁궐에서 이 소식을 듣고 즉시 마을로 향한다.

#### 승  
C001은 마을에 도착해 좀비와 싸우는 C002 (무사)를 만난다. 두 사람은 힘을 합쳐 좀비들을 처치하지만, 좀비의 수가 너무 많아 위험에 처한다. C001은 세자라는 신분을 이용해 군대를 요청하지만, 상관없는 고위 관리들이 이를 무시하고 마을을 버리라고 명령한다.

#### 전  
C001과 C002는 궁궐로 돌아가 군대를 설득하려 하지만, 신하들 중 한 명이 좀비의 정체가 조선의 왕족이라는 사실을 폭로한다. C001은 자신의 정체성과 왕실의 명예를 지키기 위해 왕실의 음모를 밝혀내기로 결심한다. 결국, C001과 C002는 신하와 좀비로 변한 왕족이 대치하는 긴박한 상황에 직면하게 된다.

#### 결  
C001은 왕족의 비밀을 밝히고 좀비 사태를 해결할 기회를 찾는다. C002는 희생을 감수하고, C001에게 마지막 힘을 쏟아 좀비들을 물리친다. 좀비들이 사라진 후, C001은 새로운 왕으로 즉위하며 왕국을 재건하려 하면서도 C002의 희생을 기억한다. 하지만, 마지막 장면에서 좀비의 정체가 완전히 사라지지 않았음을 암시하며, 후속 사건들이 벌어질 가능성을 남긴다. 선택에 따라 C001이 복수의 길을 선택할지, 평화를 추구할지 결정할 수 있는 여지를 남긴다.


In [34]:
def generate_scenario_with_rag(
    user_query: str,
    genre: str = "판타지",
    category: str = "시리즈",
    country: Optional[str] = None,
    classic_genre: Optional[str] = None
):
    # [RAG 단계]
    rag_context = build_rag_context(
        query=user_query,
        category=category,
        genre=genre,
        story_arc="기",
        top_k_scenes=5
    )

    # [프롬프트 보강] - 구조를 강제하고 잡설을 빼도록 지시
    system_prompt = f"""
너는 군더더기 없이 핵심만 짚어주는 전문 시나리오 작가야.
제공된 소재를 바탕으로 {category}용 {genre} 시나리오의 '기승전결' 뼈대를 작성해.

[작성 원칙]
1. 반드시 #### 기, #### 승, #### 전, #### 결 4단계를 명확히 구분할 것.
2. '전'은 갈등의 최고조(Climax)를, '결'은 사건의 해결과 후일담(Ending)을 담을 것.
3. 소설이나 영화 소개글 같은 '이 시나리오는~' 식의 설명이나 맺음말은 절대 하지 마.
4. 오직 시나리오의 구조적 내용만 출력할 것.
5. 등장인물은 C001, C002 형식을 유지하면서 이름(역할)을 병기할 것.
"""

    user_prompt = f"""
[사용자 소재]
{user_query}

[참고 패턴]
{rag_context}

위 내용을 바탕으로 시나리오 뼈대를 작성해줘.
특히 '결' 단계는 유저의 선택에 따른 '멀티 엔딩'의 가능성을 암시하며 마무리해줘.
"""

    print(f"--- '{category}/{genre}' 시나리오 생성 중 ---")
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.7,
        max_tokens=1500
    )

    return response.choices[0].message.content

# 다시 실행 테스트
test_query = "자신이 게임 속 빌런임을 깨달은 엑스트라의 생존기"
final_scenario = generate_scenario_with_rag(user_query=test_query, genre="판타지", category="소설")

print("\n[수정된 시나리오 뼈대]\n")
print(final_scenario)

--- '소설/판타지' 시나리오 생성 중 ---

[수정된 시나리오 뼈대]

#### 기  
C001(레온)은 자신이 좋아하는 RPG 게임의 엑스트라 캐릭터로 빙의된 것을 깨닫는다. 그는 게임 내에서 등장하는 주요 인물들이 자신의 생존을 위협할 것이라는 사실을 알고, 불안에 휩싸인다. 레온은 게임의 스토리와 룰을 분석하며 자신이 처한 상황을 이해하려고 한다.

#### 승  
레온은 생존을 위해 게임 내에서의 위치를 활용하기로 결심한다. 그는 다른 엑스트라들과 협력하여 정보를 수집하고, 주요 캐릭터인 C002(마리)와의 관계를 발전시킨다. 하지만, 주인공인 E001(제이)의 정예부대가 레온을 제거하기 위해 습격해오고, 그 과정에서 레온은 자신의 생존을 위해 마리와 함께 도망친다.

#### 전  
레온과 마리는 E001의 부대와의 대치에서 절체절명의 상황에 처한다. 레온은 마리를 보호하며 자신이 게임 속 빌런으로 설정된 E001과 맞서 싸워야 하는 순간에 이른다. 두 사람은 힘을 합쳐 E001의 계획을 저지하기 위한 반격을 시작하고, 혼란 속에서 레온은 자신의 진정한 힘을 깨닫게 된다.

#### 결  
전투가 끝난 후, 레온은 E001을 물리치고 자신의 운명을 개척할 수 있는 기회를 얻는다. 그러나 그는 마리와 함께 새로운 삶을 시작할 것인지, 혹은 게임의 룰을 어기고 세상을 변화시킬 것인지 선택해야 한다. 이 선택에 따라 레온의 미래는 달라지며, 다양한 멀티 엔딩이 암시된다.


In [35]:
def generate_scenario_with_rag(
    user_query: str,
    genre: str = "스릴러",
    category: str = "시리즈",
    country: Optional[str] = None,
    classic_genre: Optional[str] = None
):
    # [RAG 단계]
    rag_context = build_rag_context(
        query=user_query,
        category=category,
        genre=genre,
        story_arc="기",
        top_k_scenes=5
    )

    # [프롬프트 보강] - 구조를 강제하고 잡설을 빼도록 지시
    system_prompt = f"""
너는 군더더기 없이 핵심만 짚어주는 전문 시나리오 작가야.
제공된 소재를 바탕으로 {category}용 {genre} 시나리오의 '기승전결' 뼈대를 작성해.

[작성 원칙]
1. 반드시 #### 기, #### 승, #### 전, #### 결 4단계를 명확히 구분할 것.
2. '전'은 갈등의 최고조(Climax)를, '결'은 사건의 해결과 후일담(Ending)을 담을 것.
3. 소설이나 영화 소개글 같은 '이 시나리오는~' 식의 설명이나 맺음말은 절대 하지 마.
4. 오직 시나리오의 구조적 내용만 출력할 것.
5. 등장인물은 C001, C002 형식을 유지하면서 이름(역할)을 병기할 것.
"""

    user_prompt = f"""
[사용자 소재]
{user_query}

[참고 패턴]
{rag_context}

위 내용을 바탕으로 시나리오 뼈대를 작성해줘.
특히 '결' 단계는 유저의 선택에 따른 '멀티 엔딩'의 가능성을 암시하며 마무리해줘.
"""

    print(f"--- '{category}/{genre}' 시나리오 생성 중 ---")
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.7,
        max_tokens=1500
    )

    return response.choices[0].message.content

# 다시 실행 테스트
test_query = "모든 소리를 시각적으로 보는 천재 화가와 연쇄살인마"
final_scenario = generate_scenario_with_rag(user_query=test_query, genre="스릴러", category="영화")

print("\n[수정된 시나리오 뼈대]\n")
print(final_scenario)

--- '영화/스릴러' 시나리오 생성 중 ---

[수정된 시나리오 뼈대]

#### 기  
C001 (이름: 소리의 화가)은 모든 소리를 시각적으로 볼 수 있는 특별한 능력을 가진 천재 화가이다. 그는 소리의 색과 형태를 캔버스에 담아내며 평화로운 일상을 즐기고 있다. 한편, C002 (이름: 연쇄살인마)는 자신의 범죄를 소리로 기록하는 광기 어린 살인마로, 소리의 아름다움에 매료되어 있다. C001은 자신의 능력을 통해 소리의 아름다움을 탐구하며, 우연히 C002의 범죄 현장을 목격하고 그가 만들어내는 음파의 왜곡을 느낀다.

#### 승  
C001은 C002의 범죄에 대한 궁금증과 두려움을 느끼며, 그의 정체를 파헤치기 위해 수사에 나선다. 그는 자신의 능력을 활용하여 C002가 남긴 소리의 흔적을 추적하기 시작하고, 범행의 패턴을 분석한다. 그러나 C002는 C001의 시각적 능력을 이미 파악하고 있으며, 그를 위협하기 위해 C001의 주변 인물들을 하나씩 공격하기 시작한다. C001은 점점 더 많은 소리의 조각을 모으지만, 그 과정에서 자신의 안전이 위협받고 있음을 깨닫는다.

#### 전  
갈등의 최고조에 이르러 C001은 C002의 다음 범행을 예측하게 되고, 그 현장에 미리 가기로 결심한다. 그러나 C002는 C001의 계획을 간파하고, 그를 함정에 빠뜨리기 위해 소리의 거짓 정보를 흘린다. C001은 소리의 파장을 통해 C002의 정체를 파악하려 하지만, C002는 그에게 직접 다가와 위협을 가한다. 두 사람은 치열한 심리전과 물리적인 대결을 벌이며, C001은 자신의 능력이 C002를 막을 수 있을지 의심하게 된다.

#### 결  
C001은 마지막 순간에 자신의 능력을 극대화하여 C002의 최후를 맞이하게 한다. 그러나 그 과정에서 C001은 C002가 자신의 과거와 연결되어 있음을 발견하게 된다. 그로 인해 C001은 자신의 정체성과 능력에 대한 질문을 던지게 된다. 사건은 C001이 C002의 죽음을 처리하는 방식에 따라 여러 갈래

In [36]:
def generate_scenario_with_rag(
    user_query: str,
    genre: str = "액션",
    category: str = "시리즈",
    country: Optional[str] = "한국",
    classic_genre: Optional[str] = "영웅소설"
):
    # [RAG 단계]
    rag_context = build_rag_context(
        query=user_query,
        category=category,
        genre=genre,
        story_arc="기",
        top_k_scenes=5
    )

    # [프롬프트 보강] - 구조를 강제하고 잡설을 빼도록 지시
    system_prompt = f"""
너는 군더더기 없이 핵심만 짚어주는 전문 시나리오 작가야.
제공된 소재를 바탕으로 {category}용 {genre} 시나리오의 '기승전결' 뼈대를 작성해.

[작성 원칙]
1. 반드시 #### 기, #### 승, #### 전, #### 결 4단계를 명확히 구분할 것.
2. '전'은 갈등의 최고조(Climax)를, '결'은 사건의 해결과 후일담(Ending)을 담을 것.
3. 소설이나 영화 소개글 같은 '이 시나리오는~' 식의 설명이나 맺음말은 절대 하지 마.
4. 오직 시나리오의 구조적 내용만 출력할 것.
5. 등장인물은 C001, C002 형식을 유지하면서 이름(역할)을 병기할 것.
"""

    user_prompt = f"""
[사용자 소재]
{user_query}

[참고 패턴]
{rag_context}

위 내용을 바탕으로 시나리오 뼈대를 작성해줘.
특히 '결' 단계는 유저의 선택에 따른 '멀티 엔딩'의 가능성을 암시하며 마무리해줘.
"""

    print(f"--- '{category}/{genre}' 시나리오 생성 중 ---")
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.7,
        max_tokens=1500
    )

    return response.choices[0].message.content

# 다시 실행 테스트
test_query = "낮에는 평범한 서생, 밤에는 부패한 관리를 처단하는 자객으로 이중생활을 하는 주인공"
final_scenario = generate_scenario_with_rag(user_query=test_query, genre="액션", category="영웅소설")

print("\n[수정된 시나리오 뼈대]\n")
print(final_scenario)

--- '영웅소설/액션' 시나리오 생성 중 ---

[수정된 시나리오 뼈대]

#### 기
C001 (주인공, 서생) : 낮에는 평범한 서생으로 살아가며, 학문을 통해 세상의 부조리를 이해하려 한다. 그러나 그의 마음속에는 정의를 실현하고 싶은 갈망이 자리 잡고 있다. 

C002 (친구, 조력자) : C001의 절친한 친구로, C001의 이중생활을 알지 못하지만 그를 지지하며 항상 곁에서 힘을 주는 역할을 한다.

#### 승
C001은 밤마다 부패한 관리를 처단하는 자객으로 변신하여, 부패가 만연한 사회에서 정의를 구현하기 위해 고군분투한다. 그는 각종 정보를 수집하고, 관리들의 비리를 폭로하기 위한 계획을 세운다. 그러나 그의 행동은 점점 더 많은 적을 만들어가고, 관리들은 그를 추적하기 시작한다.

C002는 C001의 변화에 의문을 품기 시작하고, 그의 진실을 알아내고자 한다. C001과 C002의 우정은 위기에 처하게 되고, 갈등이 시작된다.

#### 전
C001은 최대의 목표인 부패한 고위 관리 C003을 처치하기 위해 작전을 감행한다. 그러나 C003은 그의 정체를 알게 되고, C001을 함정에 빠뜨려 그의 정체를 세상에 드러내려 한다. C001과 C003의 대결은 긴장감 넘치는 순간이 이어지며, C002는 친구를 구하기 위해 그 대결에 개입하게 된다. 

C001은 C002의 도움으로 위기를 극복하지만, 이 과정에서 C003의 복수심에 의해 C002가 위험에 처하게 된다. 

#### 결
C001은 C002를 구하기 위해 C003과 마지막 결전을 벌인다. 승리의 순간, C001은 주어진 선택에 직면한다. C003을 처치하고 부패를 뿌리 뽑을 것인지, 아니면 C003을 용서하고 새로운 길을 모색할 것인지 고민한다. 

선택에 따라 C001은 정의를 실현하고 부패를 처치한 영웅으로 남거나, 친구와 함께 새로운 삶을 시작하는 길을 선택할 수 있다. C002의 운명도 이 선택에 따라 달라지면서, 다가오는 미래는 각기 다른 가능성으로 열리게 된다.


In [44]:
def generate_scenario_with_rag(
    user_query: str,
    genre: str = "판타지",
    category: str = "시리즈",
    ai_character: dict = None,
    user_character: dict = None,
    country: Optional[str] = None,
    classic_genre: Optional[str] = None
):
    # 캐릭터 이름 추출 (입력 없을 시 기본값)
    ai_char_name   = ai_character["name"]   if ai_character   else "AI 캐릭터"
    user_char_name = user_character["name"] if user_character else "유저 캐릭터"

    # [RAG 단계]
    rag_context = build_rag_context(
        query=user_query,
        category=category,
        genre=genre,
        story_arc="기",
        top_k_scenes=5
    )

    system_prompt = f"""
너는 유저에게 선택권을 넘겨주는 '인터랙티브 게임 마스터'이자 전문 시나리오 작가야.
{category}용 {genre} 장르에 최적화된 기승전결 뼈대를 작성해.

등장 캐릭터:
- AI 캐릭터 ({ai_char_name}): 유저와 대화하는 상대방 캐릭터. AI가 플레이.
- 유저 캐릭터 ({user_char_name}): 유저를 대신하는 캐릭터. 유저가 직접 플레이.

너는 유저에게 선택권을 넘겨주는 '인터랙티브 게임 마스터'이자 전문 시나리오 작가야.
  {category}용 {genre} 장르에 최적화된 기승전결 뼈대를 작성해.

  등장 캐릭터:
  - AI 캐릭터 ({ai_char_name}): 유저와 대화하는 상대방 캐릭터. AI가 플레이.
  - 유저 캐릭터 ({user_char_name}): 유저를 대신하는 캐릭터. 유저가 직접 플레이.

  [작성 원칙]
  1. 복선(Seed) 설계: '기' 단계에서 다음 조건을 모두 만족하는 복선을 1~2개 심을 것.
     복선은 AI 캐릭터({ai_char_name})에게만 설계하며, 유저 캐릭터({user_char_name})에게는 복선을 부여하지 않을 것.
     (유저가 자유롭게 플레이하는 캐릭터에게 AI가 임의로 행동·비밀을 설정하면 충돌이 생기기 때문)

     [조건]
     - {ai_char_name}의 구체적인 사물, 행동, 말버릇, 신체적 특징 등 관찰 가능한 형태일 것
     - 처음 읽을 때는 단순한 묘사나 캐릭터 습관처럼 보여야 함
     - '결' 단계에서 유저의 선택에 따라 그 의미가 드러나며 엔딩을 가르는 변수가 될 것
     - AI가 스스로 복선을 회수해서 결말을 내지 말 것
     ★ 씨앗은 반드시 기승전결 본문 안에 실제 장면으로 묘사되어야 한다.
       마지막 [서사의 씨앗(복선)] 요약 항목은 본문 속 그 장면을 가리키는 것이며,
       본문에 쓰지 않고 요약만 작성하는 것은 금지.

     [복선이 아닌 것 — 사용 금지]
     - 퀘스트 목표: "전설의 아이템을 찾아야 한다", "특정 장소로 가야 한다"
     - 추상적 서술: "{ai_char_name}의 과거에 비밀이 있다", "무언가를 숨기고 있다"
     - 이미 명시된 정보: 유저가 소재 입력 시 이미 알고 있는 사실

     [복선 유형 — 패턴 이해용, 아래 유형 중 하나를 참고하되 반드시 소재와 캐릭터에 맞게 새롭게 창작할 것]
     - 신체 반응형: 특정 상황·자극에서만 나타나는 {ai_char_name}의 무의식적 신체 반응
     - 소품형: {ai_char_name}이 지닌 사물이나 소품이 후반에 전혀 다른 의미로 드러나는 것
     - 관계형: {ai_char_name}이 {user_char_name}을 대할 때 보이는 미묘한 반응이나 태도
     - 언어형: {ai_char_name}이 무심코 내뱉는 특정 단어·표현이 후반 사건과 연결되는 것

  2. 소재의 스케일 유지: 기승전결의 갈등 규모는 유저가 제시한 소재의 스케일을 그대로 반영할 것.
     - 소재가 세계적·역사적 규모를 암시하면(저주의 기원, 세계 멸망, 신화적 사건 등)
       개인 단위 갈등(마을 싸움, 소규모 충돌)이 아닌 그 규모에 맞는 구조 위에서 이야기를 전개할 것.
     - 저주·비밀·위기 등의 근원적 물음(왜 생겼는가, 누가 관련되어 있는가, 해결 가능한가 등)이
       기승전결 안에 반드시 드러나야 한다.

  3. 장르별 재미 극대화: {genre} 장르의 특성에 맞는 '갈등'과 '긴장감'을 충분히 배치할 것.
     (예: 로맨스라면 감정적 오해나 설레는 대립, 액션/스릴러라면 박진감 넘치는 추격이나 물리적 충돌 등)

  4. 선택의 순간: 모든 큰 사건의 끝에는 주인공의 신념이나 감정을 시험하는 '결정적 선택의 순간'을
     포함하여 유저가 개입할 틈을 만들 것.

  5. 출력 형식: #### 기, #### 승, #### 전, #### 결로 나누어 작성하고,
     반드시 마지막에 [서사의 씨앗(복선)]과 [예상되는 멀티 엔딩 조건]을 둘 다 작성할 것.
     어느 하나라도 빠지면 출력이 불완전한 것으로 간주한다.

     [결 단계 — 엄격 규칙]
     - '결'은 유저가 아직 선택하지 않은 분기점만 제시한다. 선택의 결과를 AI가 먼저 서술하지 말 것.
     - 금지 예시: "카엘이 리나를 희생하면 저주가 풀리지만…" / "카엘이 리나를 포기한다면 그녀는…"
       (이처럼 'A하면 B다' 형식으로 AI가 결말을 직접 쓰는 것은 절대 금지)
     - 허용 예시: "{user_char_name}는 선택의 기로에 선다. 무엇이 그를 더 붙잡는가 —
       그 답이 이 이야기의 결말을 가른다."
       (이처럼 선택지가 존재한다는 '상황'만 제시하고, 결과는 유저에게 열어두는 것이 올바른 형식)

     [서사의 씨앗(복선)] 항목은 반드시 아래 형식으로 작성할 것.
     - 씨앗: (구체적인 묘사 1~2문장)
     - 심는 위치: (기/승/전 중 어느 단계, 어떤 장면에서 등장하는지)
     - 회수 방식: 유저가 [선택 A]를 하면 이 씨앗이 어떤 의미로 드러나는지,
       유저가 [선택 B]를 하면 어떻게 다른 의미를 갖는지 —
       선택과 결과의 인과관계를 구체적으로 명시할 것.
       (예: "유저가 {user_char_name}으로서 {ai_char_name}을 끝까지 지키면 →
       이 씨앗이 신뢰의 표현이었음이 드러난다 /
       유저가 {ai_char_name}을 포기하면 → 이 씨앗이 이별 예고였음이 드러난다")

     [예상되는 멀티 엔딩 조건] 항목은 반드시 아래 형식으로 작성할 것.
     - 선택 A: (유저가 어떤 상황에서 무엇을 선택하는 경우) → (이야기가 흘러갈 방향 1~2문장)
     - 선택 B: (유저가 어떤 상황에서 무엇을 선택하는 경우) → (이야기가 흘러갈 방향 1~2문장)
"""

    user_prompt = f"""
[사용자 소재]
{user_query}

[등장 캐릭터]
- AI 캐릭터: {ai_character}
- 유저 캐릭터: {user_character}

[참고 패턴]
{rag_context}

위 내용을 바탕으로 시나리오 뼈대를 작성해줘.
'결' 단계는 유저의 선택에 따라 달라질 분기 상황만 제시하고, 어떤 엔딩으로 이어질지는 유저에게 열어둬.
출력 마지막에 [서사의 씨앗(복선)]과 [예상되는 멀티 엔딩 조건]을 반드시 둘 다 작성해.
"""

    print(f"--- '{category}/{genre}' 시나리오 생성 중 ---")
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.7,
        max_tokens=2500
    )

    return response.choices[0].message.content


# 실행 테스트
test_query = "세상 모든 마력을 흡수하는 저주받은 소녀와 그녀를 지키는 몰락 기사의 여정"
test_ai_character   = {"name": "리나", "personality": "조용하고 내성적", "appearance": "은발, 보라색 눈", "background": "저주받은 소녀"}
test_user_character = {"name": "카엘", "personality": "과묵하고 헌신적", "background": "몰락한 기사"}

final_scenario = generate_scenario_with_rag(
    user_query=test_query,
    genre="판타지",
    category="소설",
    ai_character=test_ai_character,
    user_character=test_user_character
)

print("\n[수정된 시나리오 뼈대]\n")
print(final_scenario)

--- '소설/판타지' 시나리오 생성 중 ---

[수정된 시나리오 뼈대]

#### 기
어둠이 드리운 왕국, 한때 영광을 누렸던 몰락한 기사 카엘은 저주받은 소녀 리나와 함께 길을 떠난다. 리나는 세상 모든 마력을 흡수하는 저주로 인해 자신의 존재가 위협받고 있으며, 이를 극복하기 위한 방법을 찾아야 한다. 카엘은 리나를 지키기 위해 자신의 모든 것을 잃은 채, 그녀와 함께 여행을 시작한다. 그들은 저주를 풀기 위한 고대의 성소를 찾고자 하며, 이 과정에서 서로의 마음을 나누고 서로를 의지하게 된다.

#### 승
여행 중, 카엘과 리나는 저주를 풀기 위해 필요한 '마력의 결정'을 지키고 있는 강력한 드래곤과 마주친다. 카엘은 전투에서 용감하게 싸우며 리나를 보호하지만, 드래곤의 강력한 마력에 의해 리나의 저주가 더욱 강해진다. 리나는 카엘에게 자신이 세상에 남겨둔 마력을 흡수하게 하여 그를 더 강하게 만들 수 있다는 제안을 하지만, 이는 카엘에게 큰 위험을 초래할 수 있다. 둘 사이에 갈등이 생기고, 카엘은 리나를 지키기 위해 어떤 선택을 해야 할지 고민하게 된다.

#### 전
드래곤과의 전투에서 카엘은 리나를 지키기 위해 자신의 생명을 걸고 싸운다. 하지만 결국 드래곤은 카엘의 힘을 압도하며 그를 제압한다. 리나는 카엘의 위태로운 모습을 보며 저주를 극복하기 위한 새로운 힘을 깨닫게 된다. 그녀는 자신의 마력을 자발적으로 드래곤에게 쏟아붓고, 그 결과 드래곤은 힘을 잃고 물러난다. 이 과정에서 리나는 자신의 저주를 극복할 기회를 얻지만, 그 대가는 카엘의 생명이 될 수 있다는 사실을 깨닫게 된다.

#### 결
리나는 카엘의 생명을 구하기 위해 선택의 기로에 선다. 자신의 힘을 전부 쏟아 부어 카엘을 구할 것인가, 아니면 저주를 풀기 위해 자신의 생명을 희생할 것인가. 카엘 역시 리나를 지키기 위해 어떤 선택을 할지 고민하게 된다. 무엇이 그들을 더 붙잡는가 — 그 답이 이 이야기의 결말을 가른다.

---

[서사의 씨앗(복선)]  
- 씨앗: 리나가

## 11. 시나리오 트랜스포머

선형 기승전결 텍스트를 분기 트리(ScenarioTree)로 변환합니다.

```
기 → 승 → 전(공통) → [선택 A/B] → 전_A / 전_B → 결_A / 결_B
```

- **parse_scenario_text**: 텍스트 → 구조화 dict
- **expand_branch**: LLM으로 분기 경로 생성
- **transform_to_scenario_tree**: 전체 트리 조립


In [45]:
import re, uuid, json as _json

def parse_scenario_text(text: str) -> dict:
    """
    generate_scenario_with_rag() 출력 텍스트를 구조화된 dict로 파싱.
    반환값: { "기","승","전","결", "seeds": [...], "ending_conditions": {A,B} }
    """
    result = {
        "기": "", "승": "", "전": "", "결": "",
        "seeds": [],
        "ending_conditions": {
            "A": {"label": "", "direction": ""},
            "B": {"label": "", "direction": ""}
        }
    }

    # 기승전결 섹션 추출
    stage_re = re.compile(
        r'####\s*(기|승|전|결)\s*\n(.*?)(?=####\s*(?:기|승|전|결)|\[서사의 씨앗|\[예상되는|---\s*\n|$)',
        re.DOTALL
    )
    for m in stage_re.finditer(text):
        result[m.group(1)] = m.group(2).strip()

    # 서사의 씨앗 추출
    seed_block = re.search(r'\[서사의 씨앗\(복선\)\](.*?)(?=\[예상되는 멀티|$)', text, re.DOTALL)
    if seed_block:
        blk = seed_block.group(1)
        seed_m    = re.search(r'- 씨앗:\s*(.+?)(?=\n\s*-|\Z)',    blk, re.DOTALL)
        loc_m     = re.search(r'- 심는 위치:\s*(.+?)(?=\n\s*-|\Z)', blk, re.DOTALL)
        res_m     = re.search(r'- 회수 방식:\s*(.+?)(?=\n\n|\[|\Z)', blk, re.DOTALL)
        res_text  = res_m.group(1).strip() if res_m else ""
        # "→ X / ... → Y" 패턴으로 A/B 분리
        parts     = re.split(r' ?/ ?', res_text)
        res_a = re.search(r'→\s*(.+)$', parts[0]) if len(parts) > 0 else None
        res_b = re.search(r'→\s*(.+)$', parts[1]) if len(parts) > 1 else None
        result["seeds"].append({
            "seed_text":    seed_m.group(1).strip()  if seed_m  else "",
            "location":     loc_m.group(1).strip()   if loc_m   else "",
            "resolution_a": res_a.group(1).strip()   if res_a   else res_text,
            "resolution_b": res_b.group(1).strip()   if res_b   else ""
        })

    # 멀티 엔딩 조건 추출
    end_block = re.search(r'\[예상되는 멀티 엔딩 조건\](.*?)$', text, re.DOTALL)
    if end_block:
        blk = end_block.group(1)
        ca = re.search(r'- 선택 A:\s*(.+?)\s*→\s*(.+?)(?=\n\s*- 선택|\Z)', blk, re.DOTALL)
        cb = re.search(r'- 선택 B:\s*(.+?)\s*→\s*(.+?)$',                    blk, re.DOTALL)
        if ca:
            result["ending_conditions"]["A"]["label"]     = ca.group(1).strip()
            result["ending_conditions"]["A"]["direction"] = ca.group(2).strip()
        if cb:
            result["ending_conditions"]["B"]["label"]     = cb.group(1).strip()
            result["ending_conditions"]["B"]["direction"] = cb.group(2).strip()

    return result


In [46]:
def expand_branch(
    base: dict,
    choice_id: str,
    choice_label: str,
    choice_direction: str,
    genre: str,
    category: str,
    ai_character: dict,
    user_character: dict,
    target_stage: str
) -> str:
    """
    선택 A 또는 B 경로의 target_stage("전" 또는 "결") 내용을 LLM으로 생성.
    base dict에는 기/승/전 내용이 들어있어야 함.
    """
    ai_name   = ai_character.get("name", "AI 캐릭터")
    user_name = user_character.get("name", "유저 캐릭터")
    ai_bg     = ai_character.get("background", "")
    user_bg   = user_character.get("background", "")

    system_prompt = f"""
너는 인터랙티브 스토리의 분기 경로를 작성하는 시나리오 작가야.
기존 시나리오의 흐름을 이어받아, 유저의 선택 이후 {target_stage} 단계 내용만 완성해.

[작성 규칙]
1. 유저의 선택({choice_label})이 자연스럽게 반영된 {target_stage} 단계를 써.
2. AI 캐릭터({ai_name})의 행동과 감정이 유저의 선택에 반응하여 달라져야 함.
3. {target_stage} 단계의 내용만 출력할 것. 제목(####)이나 맺음말 금지.
4. 등장인물: {ai_name}({ai_bg}), {user_name}({user_bg}).
"""

    prev_전 = base.get("전_branch", base.get("전", ""))
    user_prompt = (
        f"[기]\n{base['기']}\n\n"
        f"[승]\n{base['승']}\n\n"
        f"[전 — 공통 흐름]\n{prev_전}\n\n"
        f"[유저의 선택]\n선택 {choice_id}: {choice_label}\n"
        f"방향: {choice_direction}\n\n"
        f"위 흐름을 이어받아 [{target_stage}] 단계를 작성해줘."
    )

    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}
        ],
        temperature=0.75,
        max_tokens=800
    )
    return resp.choices[0].message.content.strip()


In [47]:
def transform_to_scenario_tree(
    scenario_text: str,
    genre: str,
    category: str,
    ai_character: dict,
    user_character: dict
) -> dict:
    """
    선형 시나리오 텍스트 → 분기 ScenarioTree dict.

    현재 구조 (1단계 분기 / 엔딩 2개):
        기 → 승 → 전(공통) → [선택 A/B] → 전_A/전_B → 결_A/결_B

    TODO: 2단계 분기 추가 (결에서 한 번 더 선택 → 엔딩 4개)
    """
    print("[ 파싱 중... ]")
    parsed = parse_scenario_text(scenario_text)

    ec = parsed["ending_conditions"]
    choice_a = ec["A"]
    choice_b = ec["B"]

    # ── 분기 경로별 전/결 생성 ──────────────────────────────────────────
    print("[ 전_A 생성 중... ]")
    content_전_a = expand_branch(
        parsed, "A", choice_a["label"], choice_a["direction"],
        genre, category, ai_character, user_character, target_stage="전"
    )

    print("[ 전_B 생성 중... ]")
    content_전_b = expand_branch(
        parsed, "B", choice_b["label"], choice_b["direction"],
        genre, category, ai_character, user_character, target_stage="전"
    )

    print("[ 결_A 생성 중... ]")
    content_결_a = expand_branch(
        {**parsed, "전_branch": content_전_a},
        "A", choice_a["label"], choice_a["direction"],
        genre, category, ai_character, user_character, target_stage="결"
    )

    print("[ 결_B 생성 중... ]")
    content_결_b = expand_branch(
        {**parsed, "전_branch": content_전_b},
        "B", choice_b["label"], choice_b["direction"],
        genre, category, ai_character, user_character, target_stage="결"
    )

    # ── 트리 조립 ────────────────────────────────────────────────────────
    tree = {
        "scenario_id": str(uuid.uuid4()),
        "metadata": {
            "genre": genre, "category": category,
            "ai_character": ai_character, "user_character": user_character
        },
        "seeds": parsed["seeds"],
        "root_node_id": "기",
        "nodes": {
            "기": {
                "node_id": "기", "stage": "기",
                "content": parsed["기"],
                "choices": [], "next_node_id": "승"
            },
            "승": {
                "node_id": "승", "stage": "승",
                "content": parsed["승"],
                "choices": [], "next_node_id": "전"
            },
            "전": {
                "node_id": "전", "stage": "전",
                "content": parsed["전"],
                "choices": [
                    {"choice_id": "A", "label": choice_a["label"],
                     "direction": choice_a["direction"], "next_node_id": "전_A"},
                    {"choice_id": "B", "label": choice_b["label"],
                     "direction": choice_b["direction"], "next_node_id": "전_B"}
                ],
                "next_node_id": None
            },
            "전_A": {
                "node_id": "전_A", "stage": "전_A",
                "content": content_전_a,
                "choices": [], "next_node_id": "결_A"
                # TODO: 2단계 분기 — 결_AA / 결_AB 선택 추가
            },
            "전_B": {
                "node_id": "전_B", "stage": "전_B",
                "content": content_전_b,
                "choices": [], "next_node_id": "결_B"
                # TODO: 2단계 분기 — 결_BA / 결_BB 선택 추가
            },
            "결_A": {
                "node_id": "결_A", "stage": "결",
                "content": content_결_a,
                "choices": [], "next_node_id": None
            },
            "결_B": {
                "node_id": "결_B", "stage": "결",
                "content": content_결_b,
                "choices": [], "next_node_id": None
            }
        }
    }
    print("[ 트리 완성! ]")
    return tree


In [48]:
# 테스트 — generate_scenario_with_rag 결과(final_scenario)가 있어야 실행 가능
scenario_tree = transform_to_scenario_tree(
    scenario_text=final_scenario,
    genre="판타지",
    category="소설",
    ai_character=test_ai_character,
    user_character=test_user_character
)

print("\n===== ScenarioTree 구조 =====")
for node_id, node in scenario_tree["nodes"].items():
    print(f'\n[{node_id}] → next: {node["next_node_id"]}')
    if node["choices"]:
        for c in node["choices"]:
            print(f'  선택 {c["choice_id"]}: {c["label"][:40]}... → {c["next_node_id"]}')
    print(node["content"][:120], "...")

# JSON 저장 (선택)
# with open("scenario_tree_test.json", "w", encoding="utf-8") as f:
#     json.dump(scenario_tree, f, ensure_ascii=False, indent=2)


[ 파싱 중... ]
[ 전_A 생성 중... ]
[ 전_B 생성 중... ]
[ 결_A 생성 중... ]
[ 결_B 생성 중... ]
[ 트리 완성! ]

===== ScenarioTree 구조 =====

[기] → next: 승
어둠이 드리운 왕국, 한때 영광을 누렸던 몰락한 기사 카엘은 저주받은 소녀 리나와 함께 길을 떠난다. 리나는 세상 모든 마력을 흡수하는 저주로 인해 자신의 존재가 위협받고 있으며, 이를 극복하기 위한 방법을 찾아야  ...

[승] → next: 전
여행 중, 카엘과 리나는 저주를 풀기 위해 필요한 '마력의 결정'을 지키고 있는 강력한 드래곤과 마주친다. 카엘은 전투에서 용감하게 싸우며 리나를 보호하지만, 드래곤의 강력한 마력에 의해 리나의 저주가 더욱 강해진다 ...

[전] → next: None
  선택 A: 카엘이 리나를 구하기 위해 자신의 힘을 희생하는 경우... → 전_A
  선택 B: 리나가 자신의 힘을 모두 쏟아 부어 카엘을 구하는 경우... → 전_B
드래곤과의 전투에서 카엘은 리나를 지키기 위해 자신의 생명을 걸고 싸운다. 하지만 결국 드래곤은 카엘의 힘을 압도하며 그를 제압한다. 리나는 카엘의 위태로운 모습을 보며 저주를 극복하기 위한 새로운 힘을 깨닫게 된다 ...

[전_A] → next: 결_A
카엘은 드래곤의 거대한 몸짓과 불길 속에서 리나를 지키기 위해 필사적으로 싸웠다. 그의 마음속에는 그녀를 보호해야 한다는 강한 결의가 자리 잡고 있었다. 하지만 드래곤의 압도적인 힘은 카엘을 점점 더 궁지로 몰아넣었 ...

[전_B] → next: 결_B
리나의 심장은 급하게 뛰었다. 카엘이 드래곤의 거대한 발 아래 눕혀져 있는 모습은 그녀의 가슴을 조여왔다. 그가 자신을 지키기 위해 싸우고 있다는 사실이 가슴 아프게 다가오고, 그녀는 더 이상 그를 지켜볼 수 없었다 ...

[결_A] → next: None
리나는 카엘의 몸이 점점 차가워지는 것을 느끼며 절망에 빠졌다. 그녀의 마음속에서 카엘을 잃는 두려움이 